# 01 — Load WLASL (English ASL, Top-300) and back it up to HF

Loads `Voxel51/WLASL` (~21k clips, 2,000 gloss classes, official splits), filters to the **top-300** most frequent classes (covers ~90% of usage frequency), and pushes the subset to your private HuggingFace org so the rest of the pipeline never depends on Voxel51 staying up.

**Compute:** No GPU needed. ~10 min on Colab Free CPU.

**Outputs (on Drive):**
- `data/wlasl_top300_train/`, `data/wlasl_top300_val/`, `data/wlasl_top300_test/`
- `data/wlasl_top300_test_unseen-signers/`  (subgroup eval)
- `models/gloss_vocab.json`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/dl_project'
os.makedirs(f'{BASE}/data', exist_ok=True)
os.makedirs(f'{BASE}/models', exist_ok=True)

os.environ.setdefault('HF_TOKEN', 'hf_xxx_paste_here')
HF_ORG = 'uet-signlang'   # change to your org / username

In [ ]:
!pip install -q 'datasets>=2.20' huggingface_hub
from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])

In [ ]:
from datasets import load_dataset
os.environ['HF_DATASETS_CACHE'] = f'{BASE}/data/hf_cache'

ds = load_dataset('Voxel51/WLASL')
print(ds)
print('Train example keys:', ds['train'][0].keys())

In [ ]:
from collections import Counter
import json

TOP_K = 300
counts = Counter(ds['train']['gloss'])
top_glosses = [g for g, _ in counts.most_common(TOP_K)]
print('Top-10:', top_glosses[:10])

gloss2id = {g: i for i, g in enumerate(sorted(top_glosses))}
json.dump(gloss2id, open(f'{BASE}/models/gloss_vocab.json', 'w'))
print('Saved gloss_vocab.json with', len(gloss2id), 'classes.')

In [ ]:
def to_subset(split):
    return split.filter(lambda ex: ex['gloss'] in gloss2id).map(
        lambda ex: {'label': gloss2id[ex['gloss']]})

subsets = {k: to_subset(v) for k, v in ds.items()}
for k, v in subsets.items(): print(k, len(v))

subsets['train'].save_to_disk(f'{BASE}/data/wlasl_top300_train')
if 'validation' in subsets: subsets['validation'].save_to_disk(f'{BASE}/data/wlasl_top300_val')
if 'test' in subsets:       subsets['test'].save_to_disk(f'{BASE}/data/wlasl_top300_test')

In [ ]:
# Subgroup split: test clips whose signer ID does not appear in train.
if 'test' in subsets and 'signer_id' in subsets['test'].column_names:
    train_signers = set(subsets['train']['signer_id'])
    unseen = subsets['test'].filter(lambda ex: ex['signer_id'] not in train_signers)
    print('Unseen-signer test clips:', len(unseen))
    unseen.save_to_disk(f'{BASE}/data/wlasl_top300_test_unseen-signers')
else:
    print('Skipping unseen-signer split — signer_id field not found.')

In [ ]:
# Back the subset up to your HF org so notebooks 02/03/05 don't depend on Voxel51.
from huggingface_hub import HfApi, create_repo
api = HfApi()
REPO = f'{HF_ORG}/wlasl-top300'
create_repo(REPO, repo_type='dataset', private=True, exist_ok=True)
api.upload_folder(
    folder_path=f'{BASE}/data',
    repo_id=REPO,
    repo_type='dataset',
    allow_patterns=['wlasl_top300_*/**'],
)
api.upload_file(
    path_or_fileobj=f'{BASE}/models/gloss_vocab.json',
    path_in_repo='gloss_vocab.json',
    repo_id=REPO,
    repo_type='dataset',
)
print('Pushed to', REPO)

Next → `02_build_tutor_references.ipynb`.